In [9]:
# %pip install holidays
# %pip install tensorflow
# %pip install scikit-learn
# %pip install pandas
# %pip install numpy
# %pip install lets-plot
# %pip install keras-tuner
# %pip install tensorboard

In [10]:
import pandas as pd
import numpy as np
import holidays
import joblib
import os
import keras_tuner as kt

NUM_CORES = 8

import tensorflow as tf

tf.config.threading.set_intra_op_parallelism_threads(NUM_CORES)


tf.config.threading.set_inter_op_parallelism_threads(NUM_CORES)


os.environ["OMP_NUM_THREADS"] = str(NUM_CORES)
os.environ["TF_NUM_INTRA_OP_THREADS"] = str(NUM_CORES)
os.environ["TF_NUM_INTER_OP_THREADS"] = str(NUM_CORES)

# Make numpy values easier to read.
np.set_printoptions(precision=3, suppress=True)

from tensorflow import keras
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dropout, Flatten, Dense
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import root_mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

from lets_plot import *
LetsPlot.setup_html()

# from google.colab import drive
# drive.mount('/content/drive')



In [11]:
bikes = pd.read_csv(
    "https://raw.githubusercontent.com/byui-cse/cse450-course/master/data/bikes.csv")

bikes.info()

<class 'pandas.DataFrame'>
RangeIndex: 112475 entries, 0 to 112474
Data columns (total 12 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   dteday        112475 non-null  str    
 1   hr            112475 non-null  float64
 2   casual        112475 non-null  int64  
 3   registered    112475 non-null  int64  
 4   temp_c        112475 non-null  float64
 5   feels_like_c  112475 non-null  float64
 6   hum           112475 non-null  float64
 7   windspeed     112475 non-null  float64
 8   weathersit    112475 non-null  int64  
 9   season        112475 non-null  int64  
 10  holiday       112475 non-null  int64  
 11  workingday    112475 non-null  int64  
dtypes: float64(5), int64(6), str(1)
memory usage: 11.3 MB


## Make some features

In [12]:
def create_features(df):
    # holidays
    df['dteday'] = pd.to_datetime(df['dteday'])
    dc_holidays = holidays.US(state='DC')
    df['holiday_name'] = df['dteday'].apply(lambda x: dc_holidays.get(x))
    holiday_features = pd.get_dummies(df['holiday_name'], dtype=int)
    df = pd.concat([df, holiday_features], axis=1)

    # hour of day cycles
    df["hour_sin"] = np.sin(2 * np.pi * df["hr"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hr"] / 24)

    #  year
    df['year'] = df['dteday'].dt.year

    # month cycle
    df['month'] = df['dteday'].dt.month
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

    # day of year cycle
    df['day_of_year'] = df['dteday'].dt.dayofyear
    df['day_of_year_sin'] = np.sin(2 * np.pi * df['day_of_year'] / 365)
    df['day_of_year_cos'] = np.cos(2 * np.pi * df['day_of_year'] / 365)

    # day cycle
    df['day_of_week'] = df['dteday'].dt.dayofweek.astype(int)
    df['day_of_week_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
    df['day_of_week_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)

    #  season cycle
    df['season_sin'] = np.sin(2 * np.pi * df['season'] / 4)
    df['season_cos'] = np.cos(2 * np.pi * df['season'] / 4)

    # temperature
    df['temp_diff'] = df['feels_like_c'] - df['temp_c']

    # day type
    df['day_type'] = df.apply(lambda row: 'working_day' if row['holiday'] == 0 and row['workingday'] == 1 else ('weekend' if row['workingday'] == 0 and row['holiday'] == 0 else 'holiday_weekend' if row['holiday'] == 1 and row['workingday'] == 0 else 'holiday_weekday'), axis=1)
    day_type_features = pd.get_dummies(df['day_type'], dtype=int)
    df = pd.concat([df, day_type_features], axis=1)

    # TODO: COVID-19 impact (after 2020-03-01)
    # TODO: Gas price data
    # TODO: Events


    return df.drop(columns=['holiday_name', 'month', 'day_of_week', 'temp_c', 'season', 'day_of_year', 'day_type', 'dteday', 'hr'])

In [13]:
df = create_features(bikes)

df.info()

df.describe().transpose()

<class 'pandas.DataFrame'>
RangeIndex: 112475 entries, 0 to 112474
Data columns (total 44 columns):
 #   Column                                                   Non-Null Count   Dtype  
---  ------                                                   --------------   -----  
 0   casual                                                   112475 non-null  int64  
 1   registered                                               112475 non-null  int64  
 2   feels_like_c                                             112475 non-null  float64
 3   hum                                                      112475 non-null  float64
 4   windspeed                                                112475 non-null  float64
 5   weathersit                                               112475 non-null  int64  
 6   holiday                                                  112475 non-null  int64  
 7   workingday                                               112475 non-null  int64  
 8   Christmas Day            

,count,mean,std,min,25%,50%,75%,max
casual,112475.0,90.434612,128.655621,0.000000,7.000000,3.600000e+01,1.220000e+02,1244.000000
registered,112475.0,249.193625,258.267544,0.000000,48.000000,1.800000e+02,3.600000e+02,1702.000000
feels_like_c,112475.0,14.659325,11.428324,-24.000000,5.400000,1.600000e+01,2.350000e+01,49.600000
hum,112475.0,0.636624,0.190328,0.088900,0.484100,6.409000e-01,7.988000e-01,1.000000
windspeed,112475.0,13.100614,7.857600,0.000000,7.700000,1.220000e+01,1.750000e+01,69.800000
weathersit,112475.0,1.405441,0.683450,1.000000,1.000000,1.000000e+00,2.000000e+00,4.000000
holiday,112475.0,0.030300,0.171412,0.000000,0.000000,0.000000e+00,0.000000e+00,1.000000
workingday,112475.0,0.684312,0.464791,0.000000,0.000000,1.000000e+00,1.000000e+00,1.000000
Christmas Day,112475.0,0.002561,0.050537,0.000000,0.000000,0.000000e+00,0.000000e+00,1.000000
Christmas Day (observed),112475.0,0.000854,0.029203,0.000000,0.000000,0.000000e+00,0.000000e+00,1.000000


## Train

In [14]:
X = df.drop(columns=["casual", "registered"])
y = df[["casual", "registered"]]


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


scaler_x = MinMaxScaler()
scaler_y_casual = MinMaxScaler()
scaler_y_registered = MinMaxScaler()


scaler_x.fit(X_train)
scaler_y_casual.fit(y_train[["casual"]])
scaler_y_registered.fit(y_train[["registered"]])


X_train_s = scaler_x.transform(X_train)
X_test_s = scaler_x.transform(X_test)

# For y, we transform columns individually and combine them
y_train_s = np.hstack([
    scaler_y_casual.transform(y_train[["casual"]]),
    scaler_y_registered.transform(y_train[["registered"]])
])

y_test_s = np.hstack([
    scaler_y_casual.transform(y_test[["casual"]]),
    scaler_y_registered.transform(y_test[["registered"]])
])

In [ ]:
def build_model(hp):
    model = Sequential()
    
    # 1. Tune the number of hidden layers (e.g., between 2 and 6 layers)
    for i in range(hp.Int('num_layers', 2, 6)):
        
        # 2. Tune the number of units/nodes in each layer dynamically
        model.add(Dense(
            units=hp.Int(f'units_{i}', min_value=32, max_value=256, step=32),
            # Add the new modern activations here:
            activation=hp.Choice(f'activation_{i}', values=['relu', 'leaky_relu', 'swish', 'gelu', 'mish'])
        ))
        
        # 3. Tune the dropout rate dynamically
        model.add(Dropout(rate=hp.Float(f'dropout_{i}', 0.1, 0.6, step=0.1)))
    
    # Final Output Layer (Locked to your 2 targets)
    model.add(Dense(2, activation='linear'))
    
    # Tune the learning rate of the optimizer
    hp_learning_rate = hp.Choice('learning_rate', values=[0.005, 0.0005])
    
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=hp_learning_rate),
        loss='mean_squared_error',
        metrics=['mse']
    )
    
    return model

In [16]:
tuner = kt.BayesianOptimization(
    build_model,
    objective='val_loss',
    max_trials=20,            # Total number of different models to test
    executions_per_trial=1,   # How many times to train each model configuration
    directory='tuning_dir',
    project_name='bike_sharing_nn'
)

# 1. Setup Early Stopping to kill dead trials early
stop_early = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)

# 2. Setup the Learning Rate Reducer
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss', 
    factor=0.8,       # Cut the learning rate to 80% of its current value (e.g., 0.001 -> 0.0008)
    patience=5,        # Wait 5 epochs of no improvement before dropping
    min_lr=1e-5,       # Don't let the learning rate drop lower than this
    verbose=1          # Print a note in the console when it drops
)

# 3. Pass BOTH into your tuner search
tuner.search(
    X_train_s, y_train_s, 
    epochs=1000, 
    validation_split=0.35, 
    batch_size=256, 
    callbacks=[stop_early, reduce_lr] # <-- Dynamic learning rate is now active!
)

Trial 20 Complete [00h 09m 09s]
val_loss: 0.0010783810866996646

Best val_loss So Far: 0.0009916980052366853
Total elapsed time: 05h 03m 05s


In [17]:
# 1. Print a summary of the results
tuner.results_summary()

# 2. Get the best hyperparameters
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

print(f"""
The hyperparameter search is complete.
The optimal number of layers was: {best_hps.get('num_layers')}
The optimal learning rate for the optimizer was: {best_hps.get('learning_rate')}
""")

# 3. Build the best model and train it for real
best_model = tuner.hypermodel.build(best_hps)
history = best_model.fit(X_train_s, y_train_s, epochs=2000, validation_split=0.35, batch_size=256, callbacks=[stop_early, reduce_lr])

Results summary
Results in tuning_dir\bike_sharing_nn
Showing 10 best trials
Objective(name="val_loss", direction="min")

Trial 13 summary
Hyperparameters:
num_layers: 3
units_0: 160
activation_0: relu
dropout_0: 0.1
learning_rate: 0.005
units_1: 256
activation_1: leaky_relu
dropout_1: 0.30000000000000004
units_2: 160
activation_2: leaky_relu
dropout_2: 0.30000000000000004
units_3: 256
activation_3: leaky_relu
dropout_3: 0.5
units_4: 128
activation_4: mish
dropout_4: 0.2
units_5: 128
activation_5: relu
dropout_5: 0.4
units_6: 32
activation_6: leaky_relu
dropout_6: 0.30000000000000004
Score: 0.0009916980052366853

Trial 18 summary
Hyperparameters:
num_layers: 3
units_0: 192
activation_0: mish
dropout_0: 0.2
learning_rate: 0.001
units_1: 128
activation_1: swish
dropout_1: 0.1
units_2: 256
activation_2: relu
dropout_2: 0.1
units_3: 192
activation_3: relu
dropout_3: 0.1
units_4: 160
activation_4: swish
dropout_4: 0.5
units_5: 32
activation_5: mish
dropout_5: 0.1
units_6: 128
activation_6: 

In [18]:
# 1. Create the DataFrame fresh from the history object
hist = pd.DataFrame(history.history)

# 2. Explicitly add an 'epoch' column (starting from 1)
hist['epoch'] = range(1, len(hist) + 1)

def plot_history():
    # 3. Use 'epoch' instead of 'index' for the melt
    plot_data = hist.melt(id_vars=['epoch'], 
                          value_vars=['mse', 'val_mse'], 
                          var_name='Type', 
                          value_name='MSE')

    # 4. Use 'epoch' for the x-axis
    p = ggplot(plot_data, aes(x='epoch', y='MSE', color='Type')) + \
        geom_line(size=1) + \
        labs(title="Model Training History", x="Epoch", y="Mean Square Error") + \
        theme_minimal() + \
        scale_color_discrete(labels=['Train Error', 'Val Error'])
    
    return p

# Display the plot
plot_history().show()

In [19]:
r2 = r2_score(y_test_s, best_model.predict(X_test_s))
print(f"R2 Score: {r2:.4f}")

703/703 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step
R2 Score: 0.9311


In [20]:
# 1. Extract weights from the first layer
raw_weights = best_model.layers[0].get_weights()[0]

# 2. Take the ABSOLUTE value first, then average across the neurons
feature_importance = pd.DataFrame({
    'Feature': scaler_x.feature_names_in_,
    'Importance': np.mean(np.abs(raw_weights), axis=1)  # Fixed: added np.abs()
})

# 3. Plot using lets-plot with explicit ordering
# We use as_discrete() with order_by to force the plot to stay sorted
ggplot(feature_importance, aes(x='Importance', y=as_discrete('Feature', order_by='Importance', order=1))) + \
    geom_bar(stat='identity', fill='#3b82f6') + \
    labs(title="Feature Importance (First Layer Weights)", 
         x="Average Absolute Weight Magnitude", 
         y="Feature") + \
    theme_minimal()

In [21]:
preds = best_model.predict(X_test_s)

703/703 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step


In [22]:
pred_casual_s = preds[:, 0].reshape(-1, 1)
pred_registered_s = preds[:, 1].reshape(-1, 1)


final_casual = scaler_y_casual.inverse_transform(pred_casual_s)
final_registered = scaler_y_registered.inverse_transform(pred_registered_s)


results = pd.DataFrame({
    "predicted_casual": final_casual.flatten(),
    "predicted_registered": final_registered.flatten()
})

In [23]:


# 1. Save the Keras model
# The '.keras' extension is the modern standard for TensorFlow/Keras
best_model.save("best_bike_rental_model.keras") 

# 2. Save the fitted scalers
joblib.dump(scaler_x, "scaler_x.pkl")
joblib.dump(scaler_y_casual, "scaler_y_casual.pkl")
joblib.dump(scaler_y_registered, "scaler_y_registered.pkl")

print("Model and scalers successfully saved!")

Model and scalers successfully saved!


In [24]:
# loaded_model = keras.models.load_model("bike_rental_model.keras")

# loaded_scaler_x = joblib.load("scaler_x.pkl")
# loaded_scaler_y_casual = joblib.load("scaler_y_casual.pkl")
# loaded_scaler_y_registered = joblib.load("scaler_y_registered.pkl")

# mini_df = pd.read_csv('https://raw.githubusercontent.com/byui-cse/cse450-course/master/data/biking_holdout_test_mini.csv')

# mini_X = create_features(mini_df)

# expected_columns = loaded_scaler_x.feature_names_in_

# for col in expected_columns:
#     if col not in mini_X.columns:
#         mini_X[col] = 0


# mini_X = mini_X[expected_columns]


# new_data_scaled = loaded_scaler_x.transform(mini_X)


# raw_preds = loaded_model.predict(new_data_scaled)


# pred_casual_s = raw_preds[:, 0].reshape(-1, 1)
# pred_registered_s = raw_preds[:, 1].reshape(-1, 1)


# final_casual = loaded_scaler_y_casual.inverse_transform(pred_casual_s)
# final_registered = loaded_scaler_y_registered.inverse_transform(pred_registered_s)


# results = pd.DataFrame({
#     "predicted_casual": np.round(final_casual.flatten()).astype(int),
#     "predicted_registered": np.round(final_registered.flatten()).astype(int)
# })

# results['predictions'] = results['predicted_casual'] + results['predicted_registered']

# results.drop(columns=['predicted_casual', 'predicted_registered'], inplace=True)

# results.to_csv("Sanhedrin3-predictions.csv", index=False)

In [25]:
# # 1. Extract weights from the first layer
# raw_weights = loaded_model.layers[0].get_weights()[0]

# # 2. Take the ABSOLUTE value first, then average across the neurons
# feature_importance = pd.DataFrame({
#     'Feature': loaded_scaler_x.feature_names_in_,
#     'Importance': np.mean(np.abs(raw_weights), axis=1)  # Fixed: added np.abs()
# })

# # 3. Plot using lets-plot with explicit ordering
# # We use as_discrete() with order_by to force the plot to stay sorted
# ggplot(feature_importance, aes(x='Importance', y=as_discrete('Feature', order_by='Importance', order=1))) + \
#     geom_bar(stat='identity', fill='#3b82f6') + \
#     labs(title="Feature Importance (First Layer Weights)", 
#          x="Average Absolute Weight Magnitude", 
#          y="Feature") + \
#     theme_minimal()

# If you are running live

In [26]:
# Mini holdout set

mini_df = pd.read_csv('https://raw.githubusercontent.com/byui-cse/cse450-course/master/data/biking_holdout_test_mini.csv')

mini_X = create_features(mini_df)

expected_columns = scaler_x.feature_names_in_

for col in expected_columns:
    if col not in mini_X.columns:
        mini_X[col] = 0


mini_X = mini_X[expected_columns]

mini_X = scaler_x.transform(mini_X)

preds = best_model.predict(mini_X)

pred_casual_s = preds[:, 0].reshape(-1, 1)
pred_registered_s = preds[:, 1].reshape(-1, 1)


final_casual = scaler_y_casual.inverse_transform(pred_casual_s)
final_registered = scaler_y_registered.inverse_transform(pred_registered_s)


results = pd.DataFrame({
    "predicted_casual": final_casual.flatten(),
    "predicted_registered": final_registered.flatten()
})

# Sum them and put them under the column predictions
results['predictions'] = results['predicted_casual'] + results['predicted_registered']

results.drop(columns=['predicted_casual', 'predicted_registered'], inplace=True)

results.to_csv("Sanhedrin3Tuned-predictions.csv", index=False)

12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step 
